In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Load checkpoint
ckpt = torch.load("DeepCNNLSTM_fold0_YYYYMMDD_HHMM_best.pt", map_location="cpu")

# routing_reports is a list of dicts, one per analysis epoch
reports = ckpt["routing_reports"]
print(f"Got {len(reports)} routing snapshots")
print(f"Keys in each report: {reports[0].keys()}")
# → dict_keys(['model_name', 'num_samples', 'num_experts', 'entropy',
#              'load_balance', 'routing_by_gesture', 'routing_by_pid',
#              'expert_coactivation', 'epoch'])

In [ ]:
epochs   = [r["epoch"] for r in reports]
entropy  = [r["entropy"]["mean_entropy_normalised"] for r in reports]
imbalance = [r["load_balance"]["hard_imbalance_ratio"] for r in reports]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(epochs, entropy, marker='o')
ax1.set(xlabel="Epoch", ylabel="Normalised Entropy (0=sharp, 1=flat)",
        title="Routing Entropy Over Training")
ax2.plot(epochs, imbalance, marker='s', color='orange')
ax2.axhline(1.0, linestyle='--', color='gray', label='Ideal (balanced)')
ax2.set(xlabel="Epoch", ylabel="Max/Min Imbalance Ratio", title="Load Balance")
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np

last = reports[-1]
mat = np.array(last["routing_by_gesture"]["mean_weight_matrix"])  # (G, E)
gesture_ids = last["routing_by_gesture"]["gesture_ids"]
E = last["num_experts"]

fig, ax = plt.subplots(figsize=(E+2, len(gesture_ids)))
im = ax.imshow(mat, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(E)); ax.set_xticklabels([f"E{i}" for i in range(E)])
ax.set_yticks(range(len(gesture_ids))); ax.set_yticklabels([f"G{g}" for g in gesture_ids])
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

In [ ]:
from MOE.MOE_analysis import RoutingAnalyzer, RoutingRecord
# You can also save/load the raw RoutingRecord (tensors) with:
# save_routing_record(record, "routing_record_epoch30.pt")
# record = load_routing_record("routing_record_epoch30.pt")